# 12 — Negative Conditional Entropy

Meet the sharpest signature of entanglement: a conditional entropy that drops below zero — impossible for any classical pair.

**Prerequisites:** `02/11_quantum_mutual_information`.

**What you'll be able to do**
- Compute the quantum conditional entropy $S(A|B) = S(\rho_{AB}) - S(\rho_B)$.
- Show it is $-1$ bit for a Bell pair, where classically $H(X|Y) \ge 0$ always.
- Sweep a Werner state and watch both quantum effects switch off together as noise grows.

Classical conditional entropy is never negative — knowing $Y$ can only reduce your uncertainty about $X$. Quantum mechanics violates even this. For a Bell pair, $S(A|B) = -1$ bit. We compute it, then sweep a noisy Werner state to watch the negative region shrink and vanish well before the entanglement itself does.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.quantum import quantum_mutual_information, quantum_conditional_entropy
from qot_course.quantum.composite import bell_state, tensor
from qot_course.quantum.density import density_matrix
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS
from scipy.optimize import brentq

np.random.seed(42)
viz.use_course_style()

## 1. Negative conditional entropy

The quantum conditional entropy is

$$ S(A|B) = S(\rho_{AB}) - S(\rho_B). $$

Classically the chain rule of `02/01`, $H(AB) = H(B) + H(A|B)$, makes $H(A|B)$ an average of entropies $H(A \mid B = b)$, one per value of $B$, each non-negative, so $H(A|B) \ge 0$. The quantum formula is the same difference, but there is no conditional distribution to average, and nothing holds it above zero. The correlated coins of `02/11`, $\tfrac12(|00\rangle\langle00| + |11\rangle\langle11|)$, sit exactly on the classical floor: $S(\rho_{AB}) = S(\rho_B) = 1$ bit, so $S(A|B) = 0$. For a Bell pair $S(\rho_{AB}) = 0$ (pure) but $S(\rho_B) = 1$ bit (maximally mixed reduced state), so $\mathbf{S(A|B) = -1}$ **bit** (Cerf & Adami, 1997). Its negative, $S(\rho_B) - S(\rho_{AB})$, is called the *coherent information*.

In [ ]:
rho_bell = density_matrix(bell_state())
rho_product = density_matrix(tensor(KET_0, KET_PLUS))
rho_coins = 0.5 * (density_matrix(tensor(KET_0, KET_0)) + density_matrix(tensor(KET_1, KET_1)))  # correlated coins of 02/11
print(f"S(A|B)  Bell state    = {quantum_conditional_entropy(rho_bell, dims=[2, 2]):+.3f} bit  (impossible classically)")
print(f"S(A|B)  product state = {quantum_conditional_entropy(rho_product, dims=[2, 2]):+.3f} bit")
print(f"S(A|B)  correlated coins = {quantum_conditional_entropy(rho_coins, dims=[2, 2]):+.3f} bit  (on the classical floor)")
print()
print("Classical bound:  H(X|Y) >= 0  ALWAYS.")

## 2. The Werner-state sweep

Mix the Bell state with white noise:

$$ \rho(p) = (1 - p)\,|\mathrm{Bell}\rangle\langle\mathrm{Bell}| + p\,\frac{I_4}{4}, \qquad p \in [0, 1]. $$

At $p = 0$ we have the pure Bell pair; at $p = 1$ the maximally mixed two-qubit state. We plot both $I(A{:}B)$ and $S(A|B)$ across the sweep, and mark two values of $p$: where $S(A|B)$ crosses zero, and where the state stops being entangled. For two qubits the partial transpose decides the second (Peres–Horodecki): the state is entangled exactly when the partial transpose has a negative eigenvalue.

In [ ]:
def werner_state(p):
    return (1.0 - p) * density_matrix(bell_state()) + p * (np.eye(4, dtype=complex) / 4.0)

ps = np.linspace(0.0, 1.0, 60)
qmi = np.array([quantum_mutual_information(werner_state(p), dims=[2, 2]) for p in ps])
qce = np.array([quantum_conditional_entropy(werner_state(p), dims=[2, 2]) for p in ps])

def min_eig_partial_transpose(p):
    # transpose the B index of rho[a, b, a', b'] and return the smallest eigenvalue
    pt = werner_state(p).reshape(2, 2, 2, 2).transpose(0, 3, 2, 1).reshape(4, 4)
    return np.linalg.eigvalsh(pt).min()

p_zero = brentq(lambda p: quantum_conditional_entropy(werner_state(p), dims=[2, 2]), 0.05, 0.95)
p_ppt = brentq(min_eig_partial_transpose, 0.05, 0.95)
print(f"S(A|B) crosses zero at p = {p_zero:.4f}")
print(f"partial transpose has a negative eigenvalue up to p = {p_ppt:.4f}")
print(f"at p = 0.5: S(A|B) = {quantum_conditional_entropy(werner_state(0.5), dims=[2, 2]):+.3f} bit, "
      f"smallest partial-transpose eigenvalue = {min_eig_partial_transpose(0.5):+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True)

axes[0].plot(ps, qmi, color=viz.SOURCE_COLOR, lw=2)
axes[0].axhline(1.0, color=viz.TARGET_COLOR, linestyle="--", label="classical max (1 bit)")
axes[0].axhline(0.0, color=COLORS["muted"], linewidth=0.8)
axes[0].set_xlabel("Werner noise parameter  p")
axes[0].set_ylabel("I(A:B)  (bits)")
axes[0].set_title("Quantum mutual information", pad=10)
axes[0].legend(loc="upper right")

axes[1].plot(ps, qce, color=viz.FLOW_COLOR, lw=2)
axes[1].axhline(0.0, color=viz.TARGET_COLOR, linestyle="--", label="classical floor (0)")
axes[1].fill_between(ps, qce, 0.0, where=qce < 0.0, color=viz.FLOW_COLOR, alpha=0.18,
                     label="S(A|B) < 0")
axes[0].axvline(p_ppt, color=COLORS["muted"], linestyle=":", lw=1.4)
axes[1].axvline(p_ppt, color=COLORS["muted"], linestyle=":", lw=1.4, label="entangled up to p = 2/3")
axes[1].set_xlabel("Werner noise parameter  p")
axes[1].set_ylabel("S(A|B)  (bits)")
axes[1].set_title("Quantum conditional entropy", pad=10)
axes[1].legend(loc="lower right")

fig.suptitle("Werner-state sweep: two purely-quantum effects", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Read both panels.**

- **Left.** $I(A{:}B)$ starts at $2$ bits (pure Bell, double the classical maximum) and decays to $0$ as the state becomes the maximally mixed $I/4$. Everything above the dashed amber line ($1$ bit) is quantum.
- **Right.** $S(A|B)$ starts at $-1$ bit (negative — impossible classically) and rises through zero at $p = 0.252$. The shaded region, $S(A|B) < 0$, is where the conditional entropy certifies entanglement, because no separable state ever goes below zero (Nielsen & Kempe 2001, the inequality $S(AB) \ge S(B)$ that `02/11` used for its classical ceiling). It is not the whole entangled zone: the dotted line at $p = 2/3$ is where the partial transpose loses its negative eigenvalue, so every state between $p = 0.252$ and $p = 2/3$ is entangled with $S(A|B) \ge 0$ (at $p = 0.5$, $S(A|B) = +0.549$ bit while the partial transpose still has the eigenvalue $-0.125$).

Two purely quantum effects, one sweep, shared origin — **entanglement**. As $p$ grows both quantities return to their classical range at the same $p$ (exercise 2 shows why), well before the entanglement itself is gone.

*Caveat.* $S(A|B) < 0$ is *sufficient* for entanglement but not *necessary*: this very sweep holds entangled states with $S(A|B) \ge 0$. A negative conditional entropy is a witness you can trust when it fires, and its silence proves nothing.

## Your turn

1. **The threshold by hand.** The Werner state has eigenvalues $1 - \tfrac34 p$ (once) and $\tfrac p4$ (three times), and $\rho_B = I/2$ for every $p$. Write $S(A|B)$ as a function of $p$ from these numbers alone, solve $S(A|B) = 0$ with a root finder, and recover the value printed above. Then write the eigenvalues of the partial transpose and show that the smallest is $\tfrac34 p - \tfrac12$, negative exactly when $p < 2/3$.
2. **Why the panels cross together.** Show that along the sweep $I(A{:}B) = 1 - S(A|B)$ bit, so $I(A{:}B)$ falls through $1$ bit at the same $p$ where $S(A|B)$ crosses zero. Which property of the Werner state makes this hold? Build a two-qubit state whose reduced states are not maximally mixed and check that the two crossings then come apart.
3. **Pure-state CMI.** For a pure entangled state $\cos\theta\,|00\rangle + \sin\theta\,|11\rangle$, show that $S(A|B) = -S(\rho_A) \le 0$ for all $\theta$.

## What you built

- You computed the quantum conditional entropy and found $-1$ bit for a Bell pair.
- You swept a Werner state and watched both quantum effects (excess MI, negative CMI) switch off as noise grew.
- You met the honest caveat: $S(A|B) < 0$ certifies entanglement, but the Werner states between $p = 0.252$ and $p = 2/3$ are entangled with $S(A|B) \ge 0$.

## What's next

The information arc has one piece left: a *distance* between density matrices. The synthesis `02/13_bures_distance` lifts the Fisher–Rao distance of `02/08` to the quantum setting — the Bures distance — and completes Module 02, the spine of the course.

## References

- N. J. Cerf & C. Adami, "Negative entropy and information in quantum mechanics", *Physical Review Letters* 79, 5194 (1997). DOI:10.1103/PhysRevLett.79.5194
- M. A. Nielsen & J. Kempe, "Separable states are more disordered globally than locally", *Physical Review Letters* 86, 5184–5187 (2001). DOI:10.1103/PhysRevLett.86.5184
- M. M. Wilde, *Quantum Information Theory*, ch. 11, Cambridge University Press (2017).

**Previous:** `notebooks/02_InformationAndGeometry/11_quantum_mutual_information.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/13_bures_distance.ipynb`